# Prompt Optimization with Tool Calling — with a Response Formatting Primer
> **Reference Tutorial:** This notebook accompanies the *Prompt Optimization with Tool Calling* tutorial on SAP Developers, and adds a short primer on SAP AI Core's **Response Formatting** feature so you understand how structured output can be enforced in two complementary ways.
> Run the cells **in order, top to bottom** — later cells depend on variables created earlier (`client`, `configuration_id`, `execution_id`, etc.).

---

## 👋 Who this notebook is for

You don't need prior experience with SAP AI Core's **Prompt Optimization** or **Orchestration** services to follow along. You *should* be comfortable with:
- Basic Python (functions, dictionaries, loops)
- Making REST API calls (we use the `requests` library throughout)
- Having a working SAP AI Core / Generative AI Hub tenant with credentials

If any SAP-specific term below is unfamiliar, check the **Glossary** — every step also re-explains the concept in context as it comes up.

## 🧠 What problem are we solving?

Large language models (LLMs) are often asked to look at a user's question and decide **which tools/functions to call** and **with what arguments** ("tool calling" / "function calling"). The quality of the answer depends heavily on how the **system prompt** is written — a vague prompt like `"You are a helpful assistant."` often makes the model respond in plain English instead of structured JSON, which breaks any downstream code expecting machine-readable tool calls.

There are two complementary ways SAP AI Core lets you push a model toward reliable, structured output:

1. **Prompt optimization** (the main subject of this notebook) — automate the trial-and-error of prompt engineering. Give SAP AI Core a starting prompt, a labeled dataset, and a scoring metric, and let an optimizer iteratively rewrite the prompt until it reliably produces the output shape you want.
2. **Response Formatting** (a short primer added right after Step 1) — a built-in **Orchestration Service** parameter that constrains the model's output at the *API level* — independent of prompt wording — using one of three modes: plain text, a generic JSON object, or a strict JSON Schema.

These aren't competing techniques — they solve the same problem from different layers, and in practice you'd often use them **together**: an optimized prompt to make the model's *reasoning* reliably produce the right structure, plus a `response_format` schema to *guarantee* the API-level output is parseable even if the model drifts. The primer section explains this pairing in more detail.

## 🗺️ Pipeline at a glance

```
 ┌─────────────────┐     ┌──────────────────┐     ┌────────────────────┐
 │ 1. Connect to    │ --> │ 📄 Response       │ --> │ 2. Configure         │
 │    AI Core       │     │  Formatting primer │     │  optimization params │
 └─────────────────┘     └──────────────────┘     └────────────────────┘
                                                              │
                                                              v
 ┌─────────────────┐     ┌──────────────────┐     ┌────────────────────┐
 │ 6. Register       │ <-- │ 5. Register base  │ <-- │ 3–4. Load/normalize + │
 │  dataset artifact │     │  prompt template   │     │  upload BFCL data      │
 └─────────────────┘     └──────────────────┘     └────────────────────┘
         │
         v
 ┌─────────────────┐     ┌──────────────────┐     ┌────────────────────┐
 │ 7. Run + monitor │ --> │ 8. Retrieve the    │ --> │ 9. Compare base vs   │
 │  the execution    │     │  optimized prompt   │     │  optimized (live)     │
 └─────────────────┘     └──────────────────┘     └────────────────────┘
```

## 📖 Glossary (SAP AI Core terms used throughout)

| Term | Meaning |
|---|---|
| **AI Core** | SAP's managed platform for running AI workloads (training, inference, orchestration) in the cloud. |
| **Generative AI Hub (`gen_ai_hub`)** | The Python SDK / proxy layer that lets you call LLMs and AI Core services with a unified API, regardless of the underlying model provider. |
| **Resource group** | A logical partition inside an AI Core tenant used to isolate resources between teams or projects. |
| **Scenario** | A named "workflow type" registered in AI Core (e.g. `genai-optimizations`) that groups related artifacts, configurations, and executions together. |
| **Artifact** | A registered reference to a piece of data (here: a folder of dataset files) that AI Core executions can read as input. |
| **Prompt Registry** | A versioned store for prompt templates, referenced by name+version, updatable by the optimizer without you managing raw text files. |
| **Configuration** | A saved combination of parameters (metric, models, dataset filenames, prompt reference, etc.) that fully describes *how* an optimization run should behave — but doesn't run it yet. |
| **Execution** | An actual *run* of a configuration — the long-running job that performs the optimization and produces a result. |
| **Golden record** | One row of your evaluation dataset: an input question plus the *correct* expected output, used to score candidate prompts. |
| **BFCL v3** | The [Berkeley Function-Calling Leaderboard](https://gorilla.cs.berkeley.edu/leaderboard.html) dataset (v3) — a benchmark of questions paired with correct function/tool calls, used here as training/test data. |
| **`JSON_Match`** | A built-in optimization metric that structurally compares a candidate's JSON output against the golden answer. |
| **Response Formatting** | An Orchestration Service parameter (`response_format`) that constrains model output to plain text, a JSON object, or a strict JSON Schema — enforced at the API level, independent of prompt wording. |
| **Orchestration Service** | The AI Core service used to run live inference against a deployed model, optionally chaining prompt templates, grounding, response formatting, and other modules. |

> ⚠️ **Prerequisites:** Ensure your `.env` file is configured with `AICORE_BASE_URL`, `AICORE_AUTH_URL`, `AICORE_CLIENT_ID`, `AICORE_CLIENT_SECRET`, and `AICORE_RESOURCE_GROUP` before running this notebook. You'll also need the BFCL v3 dataset file (`BFCL_v3_parallel_multiple_10tools.json`) in the same directory.

---

## Step 1 — Environment Variables Setup & Connect to AI Core

**What this step does:**
Loads credentials from the `.env` file and initializes the `GenAIHubProxyClient` — the main entry point for all AI Core API calls in Python. Every later step re-uses this single `client` object (either directly, or via `client.ai_core_client.base_url` / `client.request_header` to build raw REST calls).

**Why a `.env` file instead of hard-coding credentials?**
Client ID/secret are sensitive. Keeping them in a `.env` file (loaded via `python-dotenv`) means they never get committed to source control or pasted into a notebook that might be shared.

**Create a `.env` file** in the same directory as this notebook with the following content:

```env
AICORE_CLIENT_ID=<your client id>
AICORE_CLIENT_SECRET=<your client secret>
AICORE_AUTH_URL=<your auth url>
AICORE_BASE_URL=<your base url>
AICORE_RESOURCE_GROUP=<your resource group>
```

**Where do these values come from?** They're generated when you create a **service key** for your AI Core service instance in the SAP BTP cockpit — the JSON service key contains `clientid`, `clientsecret`, the OAuth `url` (→ `AICORE_AUTH_URL`), and the API `serviceurls.AI_API_URL` (→ `AICORE_BASE_URL`).

> 💡 No S3 or AWS credentials are required for this flow — all files are uploaded directly to AI Core's built-in dataset storage via the `/lm/dataset/files` endpoint (more on this in Step 4).

In [1]:
from collections import defaultdict
from gen_ai_hub.proxy.gen_ai_hub_proxy import GenAIHubProxyClient
from dotenv import load_dotenv
import os
import json
import requests
import random
from urllib.parse import quote
from pathlib import Path
from typing import List, Tuple
import time
from ai_api_client_sdk.models.parameter_binding import ParameterBinding
from ai_api_client_sdk.models.input_artifact_binding import InputArtifactBinding
from pydantic import BaseModel
from ai_api_client_sdk.models.artifact import Artifact

load_dotenv(override=True)

# ── SAP AI Core client ────────────────────────────────────────────────────────
client = GenAIHubProxyClient(
    base_url=os.getenv("AICORE_BASE_URL"),
    auth_url=os.getenv("AICORE_AUTH_URL"),
    client_id=os.getenv("AICORE_CLIENT_ID"),
    client_secret=os.getenv("AICORE_CLIENT_SECRET"),
    resource_group=os.getenv("AICORE_RESOURCE_GROUP")
)
resource_group = client.request_header[
    client.ai_core_client.rest_client.resource_group_header
]

print("✅ Connected to AI Core")
print(f"   Resource group: {resource_group}")

✅ Connected to AI Core
   Resource group: grounding


**Reading the output:** `✅ Connected to AI Core` confirms the OAuth handshake succeeded and `client` is ready to use. The **resource group** printed (`grounding` in this run) is the namespace all subsequent artifacts, configurations, prompts, and executions will be created inside.

---

## 📄 Background Primer — Response Formatting in SAP AI Core

Before diving into prompt optimization, it's worth understanding a related, complementary feature: **Response Formatting**, part of the Orchestration Service's `prompt_templating` module. It's documented on the SAP Help Portal under *Generative AI Hub → Orchestration → Response Formatting* ([help.sap.com](https://help.sap.com/docs/sap-ai-core/generative-ai/response-formatting)).

### What it is

When you send a request through the Orchestration Service, you can attach a `response_format` setting to the prompt template. This tells AI Core how to constrain the shape of the model's output, independent of anything you write in the system/user prompt text. There are three modes:

| Mode | What it does | When to use it |
|---|---|---|
| **`text`** | The default — the model's output is plain, unstructured text. | General conversational responses, summaries, free-form writing. |
| **`json_object`** | The model's output is structured as a generic JSON object, but the *shape* of that object (which keys, which types) isn't enforced. | You need JSON for easy parsing, but the exact structure can vary or isn't critical to pin down in advance. |
| **`json_schema`** | The model's output is validated against a JSON Schema you provide (property names, types, required fields). This is the strictest mode. | You need guaranteed, strict data validation — e.g. feeding the output directly into a downstream system that expects an exact contract. |

### How it looks in code

Using the same `gen_ai_hub.orchestration` SDK this notebook already relies on for live inference (see Step 9), a `json_schema`-constrained request looks like this:

```python
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from gen_ai_hub.orchestration.models.response_format import ResponseFormatJsonSchema

json_schema = {
    "title": "Person",
    "type": "object",
    "properties": {
        "firstName": {"type": "string", "description": "The person's first name."},
        "lastName":  {"type": "string", "description": "The person's last name."}
    }
}

template = Template(
    messages=[
        SystemMessage("You are a helpful translation assistant."),
        UserMessage("{{?user_query}}")
    ],
    response_format=ResponseFormatJsonSchema(
        name="person", description="person mapping", schema=json_schema
    ),
    defaults=[TemplateValue(name="user_query", value="Who was the first person on the moon?")]
)
# Response:
# { "firstName": "Neil", "lastName": "Armstrong" }
```

Swapping in `ResponseFormatText()` or `ResponseFormatJsonObject()` instead of `ResponseFormatJsonSchema(...)` selects the `text` or `json_object` modes respectively — same `Template` object, just a different `response_format` argument.

### How this connects to the rest of this notebook

Everything from Step 2 onward relies on **prompt engineering alone** to get structured tool-call JSON out of the model — the base prompt starts as `"You are a helpful assistant."`, and the optimizer rewrites the *wording* of that prompt until the model reliably emits valid JSON (you'll see this validated with a hand-rolled `json.loads()` check in Step 9's `compare_prompts` function). That approach works, but it depends entirely on the model choosing to follow the prompt's instructions — nothing at the API level *forces* valid JSON.

`response_format=json_schema` addresses that gap from the other direction: instead of relying on prompt wording to *persuade* the model into the right shape, it constrains the API response itself. In production, the strongest setup often combines both:
- **Prompt optimization** (this notebook) → makes the model's *reasoning process* reliably arrive at correct tool names and arguments.
- **`response_format=json_schema`** → guarantees the *wire format* of the response is always valid JSON matching your schema, even in edge cases where the model might otherwise wrap output in markdown fences or add stray commentary.

You could take the `bfcl-tool-optimized-gemini25:0.0.1` prompt this notebook produces in Step 8, and additionally attach a `json_schema` response format (built from the unioned tool definitions in `bfcl_tools.json` from Step 3) to your production `OrchestrationConfig` calls, for defense-in-depth structured output.

> 💡 Further reading: [SAP Help Portal — Response Formatting](https://help.sap.com/docs/sap-ai-core/generative-ai/response-formatting) and the [Orchestration Service SDK reference](https://help.sap.com/doc/generative-ai-hub-sdk/CLOUD/en-US/_reference/orchestration-service.html).

---

## Step 2 — Configure Optimization Parameters

**What this step does:**
Defines all configuration constants used throughout the notebook — dataset path, prompt name/version, reference model, target model, metric, and the Pydantic models for the prompt template spec.

**"Reference" vs "target" model — what's the difference?**
- The **reference model** (`REFERENCE_MODEL`, e.g. `gpt-4o:2024-08-06`) acts as a *teacher* the optimizer can compare against while it searches for a better prompt.
- The **target model(s)** (`TARGET_MODELS`) are the model(s) the final optimized prompt is actually being tuned *for*. Different models respond differently to identical prompt wording, so a prompt optimized for Gemini 2.5 Pro may look different from one optimized for GPT-4o.

**Why split into train/test samples?**
- **Train samples** (`N_TRAIN_SAMPLES = 25`) are what the optimizer actively uses to *generate and refine* candidate prompts.
- **Test samples** (`N_TEST_SAMPLES = 15`) are held out and only used to *score* each candidate prompt, so the reported score reflects genuine generalization rather than memorization — the same train/test split idea used in traditional ML.

**Why `JSON_Match` here?** This notebook uses the built-in `JSON_Match` metric, which does a structural/string comparison between the candidate's JSON output and the golden answer. It's simpler to set up than a custom LLM-as-a-judge metric (no separate metric-creation step required) and works well when you mainly care about exact structural correctness rather than nuanced partial credit.

**Key parameters:**
| Parameter | Value | Description |
|---|---|---|
| `REFERENCE_MODEL` | `gpt-4o:2024-08-06` | Teacher model used for evaluation |
| `TARGET_MODELS` | `gemini-2.5-pro:001` | Model to optimize the prompt for |
| `METRIC` | `JSON_Match` | Evaluates whether tool call JSON matches the golden answer |
| `N_TRAIN_SAMPLES` | 25 | Number of samples used to train/refine the prompt |
| `N_TEST_SAMPLES` | 15 | Number of samples used to evaluate candidate prompts |

**What are the Pydantic models for?**
`PromptTemplateMsg` and `PromptTemplateSpec` give the prompt template a strict, typed shape (a list of `{role, content}` messages) before it's serialized to JSON and pushed to the Prompt Registry in Step 5. Using Pydantic here catches typos/shape errors locally instead of getting a cryptic 400 error from the API.

> ⚠️ Verify that `gpt-4o:2024-08-06` and `gemini-2.5-pro:001` are available in your AI Core tenant before running. Check via Generative AI Hub → Models.

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
BFCL_DATASET      = "BFCL_v3_parallel_multiple_10tools.json"
BFCL_DATASET_MODE = "parallel_multiple"
N_TRAIN_SAMPLES   = 25
N_TEST_SAMPLES    = 15

PROMPT_NAME    = "bfcl-tool-base"
PROMPT_VERSION = "0.0.1"
SCENARIO       = "genai-optimizations"

SYSTEM_PROMPT   = "You are a helpful assistant."
PROMPT_TEMPLATE = "{{?question}}"
FIELDS          = ["question"]

# Use a model confirmed available in your region
REFERENCE_MODEL = "gpt-4o:2024-08-06"
TARGET_MODELS = {
    "gemini-2.5-pro:001": "bfcl-tool-optimized-gemini25:0.0.1",
}
METRIC = "JSON_Match"

# ── Pydantic models ───────────────────────────────────────────────────────────
class PromptTemplateMsg(BaseModel):
    role: str
    content: str

class PromptTemplateSpec(BaseModel):
    template: List[PromptTemplateMsg]

prompt = PromptTemplateSpec(template=[
    PromptTemplateMsg(role="system", content=SYSTEM_PROMPT),
    PromptTemplateMsg(role="user",   content=PROMPT_TEMPLATE),
])

print("✅ Configuration set")
print(f"   Dataset       : {BFCL_DATASET}")
print(f"   Scenario      : {SCENARIO}")
print(f"   Prompt name   : {PROMPT_NAME}:{PROMPT_VERSION}")
print(f"   Reference     : {REFERENCE_MODEL}")
print(f"   Target        : {list(TARGET_MODELS.keys())}")
print(f"   Metric        : {METRIC}")

✅ Configuration set
   Dataset       : BFCL_v3_parallel_multiple_10tools.json
   Scenario      : genai-optimizations
   Prompt name   : bfcl-tool-base:0.0.1
   Reference     : gpt-4o:2024-08-06
   Target        : ['gemini-2.5-pro:001']
   Metric        : JSON_Match


**Reading the output:** this echoes back the configuration you set, so you can sanity-check it before anything gets created remotely.

---

## Step 3 — Load and Normalize the BFCL v3 Dataset

**What this step does:**
Loads the BFCL v3 dataset file and normalizes it into the SAP optimizer "golden format." The optimizer doesn't understand BFCL's native structure — it expects each example as a **golden record**: `{"fields": {"question": ...}, "answer": "<JSON string>"}`. So this step is a translation layer between "the format the benchmark ships in" and "the format the optimizer needs."

Several helper functions work together to do this translation. We've split the dense original code into smaller, labeled pieces below so each piece's job is clear:

1. **`read_bfcl_file`** — a robust reader that handles 3 BFCL file formats: JSON array, standard JSONL, and concatenated JSON objects (the native BFCL v3 format).
2. **`normalize_bfcl_tool`** — converts raw BFCL tool definitions to OpenAI ChatCompletions format (e.g., `float` → `number`, `dict` → `object`, `any` → `string`). These normalized tool schemas are also exactly the kind of thing you'd feed into `ResponseFormatJsonSchema` from the primer above, if you wanted to enforce output structure at the API level too.
3. **`dedupe_tool_name`** / **`union_bfcl_tools`** — deduplicates tools with identical names but different schemas across samples.
4. **`detect_tool_key`** / **`detect_question_key`** — auto-detects the correct field names in the dataset.
5. **`build_golden`** — converts each BFCL sample into a SAP optimizer golden record:
   - `fields.question` → the user query text
   - `answer` → a JSON object string of merged tool calls (e.g. `{"weather_forecast": {...}, "calculate_distance": {...}}`)

**Output format per golden record:**
```json
{
  "fields": { "question": "I'm planning a trip to Japan..." },
  "answer": "{\"currency_conversion\": {\"amount\": [5000.0], ...}, \"calculate_distance\": {...}}"
}
```

> 💡 The `answer` must be a JSON **object** string (not an array), where each key is a tool name and each value is its arguments dict.

### 3.1 — Robustly reading the raw BFCL file

BFCL v3's native file format isn't a single JSON array or clean JSONL — it's a stream of back-to-back JSON objects with no separators (`{...}{...}{...}`), which trips up a plain `json.load()`. `read_bfcl_file` tries three strategies in order and falls back gracefully:

1. Is the whole file one JSON array `[ {...}, {...} ]`? Parse it directly (and un-wrap double-encoded strings if needed).
2. Is it standard JSONL (one JSON object per line)? Parse line-by-line.
3. Otherwise, assume it's concatenated JSON objects and use Python's `json.JSONDecoder().raw_decode()` in a loop to scan through the text character-by-character, pulling out one valid object at a time — this is what actually handles native BFCL v3 files.

In [3]:
# ── BFCL v3 file reader ───────────────────────────────────────────────────────
def read_bfcl_file(file_path: Path) -> list:
    """Robust reader for BFCL v3 files (concatenated JSON objects)."""
    with open(file_path, "r") as f:
        content = f.read().strip()

    print(f"File size: {len(content):,} bytes")

    # Format 1: JSON array [ {...}, {...} ]
    if content.startswith("["):
        try:
            result = json.loads(content)
            if result and isinstance(result[0], str):
                print("Detected double-encoded strings — decoding...")
                result = [json.loads(item) for item in result]
            print(f"Loaded as JSON array: {len(result)} records")
            return result
        except json.JSONDecodeError:
            pass

    # Format 2: Standard JSONL — one complete object per line
    if "\n" in content:
        objects = []
        for line in content.split("\n"):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    objects.append(obj)
            except json.JSONDecodeError:
                pass
        if objects:
            print(f"Loaded as JSONL: {len(objects)} records")
            return objects

    # Format 3: Concatenated/multi-line JSON objects — use raw_decode to scan through
    # This is the correct format for BFCL v3: {...}\n{...}\n{...}
    objects = []
    decoder = json.JSONDecoder()
    idx = 0
    while idx < len(content):
        while idx < len(content) and content[idx] in " \t\n\r":
            idx += 1
        if idx >= len(content):
            break
        if content[idx] != "{":
            idx += 1
            continue
        try:
            obj, end_idx = decoder.raw_decode(content, idx)
            if isinstance(obj, dict):
                objects.append(obj)
            idx = end_idx
        except json.JSONDecodeError:
            idx += 1
            continue

    print(f"Loaded as concatenated JSON: {len(objects)} records")
    return objects


### 3.2 — Normalizing tool/function schemas

BFCL tool definitions use its own type vocabulary (`float`, `dict`, `any`) that doesn't match the OpenAI-style function-calling schema most LLM proxies expect (`number`, `object`, `string`). `normalize_bfcl_tool` walks each tool's parameter definitions and remaps these types, and also sanitizes tool names (`reformat_tool_name` replaces dots with underscores, since dotted names aren't valid function identifiers for most providers). The result is wrapped in the `{"type": "function", "function": {...}}` envelope that most chat-completion APIs expect.

In [4]:
# ── BFCL v3 normalisation ─────────────────────────────────────────────────────
def reformat_tool_name(name: str) -> str:
    return name.replace(".", "_")

def normalize_bfcl_tool(tool: dict) -> dict:
    """Convert raw BFCL tool definition to OpenAI ChatCompletions format."""
    tool = json.loads(json.dumps(tool))       # deep copy
    tool["parameters"]["type"] = "object"     # BFCL uses 'dict'
    tool["name"] = reformat_tool_name(tool["name"])
    for param in tool["parameters"].get("properties", {}).values():
        if param["type"] == "float":
            param["type"] = "number"
        elif param["type"] == "dict":
            param["type"] = "object"
        elif param["type"] == "any":
            param["type"] = "string"
        elif param["type"] == "array" and "items" in param:
            if param["items"].get("type") == "float":
                param["items"]["type"] = "number"
            elif param["items"].get("type") == "dict":
                param["items"]["type"] = "object"
    return {"type": "function", "function": tool}



### 3.3 — Deduplicating tools and detecting field names across samples

A single dataset file can define the *same-named* tool slightly differently across different samples. `union_bfcl_tools` walks every sample, and if it finds a tool name reused with a genuinely different schema, it renames the newer one (`dedupe_tool_name` appends a suffix like `_n2`) so both variants can coexist in the unioned tool list without silently overwriting each other — and it prints a `WARNING` whenever this happens, which is useful for auditing data quality.

`detect_tool_key` and `detect_question_key` are small defensive helpers: instead of hard-coding `sample["function"]` and `sample["question"]`, they check a short list of plausible key names first. This makes the loader resilient if you swap in a differently-named BFCL variant later.

In [5]:
def dedupe_tool_name(tool_name: str, answers: list, occurrences: int):
    new_name = f"{tool_name}_n{occurrences}"
    new_answers = []
    for answer in answers:
        new_answer = {}
        for k, v in answer.items():
            new_answer[reformat_tool_name(new_name if k == tool_name else k)] = v
        new_answers.append(new_answer)
    return new_name, new_answers

def union_bfcl_tools(samples: list, tool_key: str = "function") -> list:
    """Union and deduplicate all tools across samples into one normalized list."""
    tool_map = {}
    tool_name_occurrences = defaultdict(int)
    tool_name_to_definition = {}
    for sample in samples:
        for tool in sample[tool_key]:
            tool_name = tool["name"]
            tool_name_occurrences[tool_name] += 1
            is_duplicate = (
                tool_name in tool_name_to_definition and
                tool != tool_name_to_definition[tool_name]
            )
            if is_duplicate:
                new_name, new_answers = dedupe_tool_name(
                    tool_name, sample["answer"], tool_name_occurrences[tool_name]
                )
                print(f"WARNING: duplicate tool {tool_name!r} → renamed to {new_name!r}")
                tool["name"]     = new_name
                sample["answer"] = new_answers
            tool_name_to_definition[tool["name"]] = tool
            normalized = normalize_bfcl_tool(tool)
            tool_map[normalized["function"]["name"]] = normalized
    return list(tool_map.values())

def detect_tool_key(sample: dict) -> str:
    """Detect which key holds the tool definitions in a BFCL sample."""
    for candidate in ["function", "functions", "tools", "tool"]:
        if candidate in sample:
            return candidate
    raise KeyError(
        f"Cannot find tool key in sample. Available keys: {list(sample.keys())}"
    )

def detect_question_key(sample: dict) -> str:
    """Detect which key holds the question/messages in a BFCL sample."""
    for candidate in ["question", "messages", "turns", "prompt"]:
        if candidate in sample:
            return candidate
    raise KeyError(
        f"Cannot find question key in sample. Available keys: {list(sample.keys())}"
    )



### 3.4 — Building golden records and running the full load

`build_golden` takes one raw BFCL sample and produces exactly the `{"fields": ..., "answer": ...}` shape the optimizer needs:
- It joins multi-turn question text into a single string.
- It merges the (potentially multiple) tool-call dicts BFCL provides per sample into one flat `merged_answer` dict, keyed by tool name.
- It flattens a BFCL quirk where some values arrive as a nested single-element list, and stringifies floats so they match the normalized string-typed schema.

`load_bfcl_dataset` ties everything together: read the file → randomly sample `n_train + n_test` records (with a fixed `random.seed(42)` for reproducibility) → detect field names → union/normalize the tools → build golden records for the train and test splits.

**Reading the output below:** you should see the file size, which format was auto-detected, the detected key names, the resulting train/test counts, how many *unique* normalized tools were unioned, and finally a full example of one golden record — this is your chance to visually sanity-check that the question and answer look correct before uploading anything.

In [6]:
def build_golden(sample: dict, question_key: str = "question") -> dict:
    """Convert one BFCL sample to a SAP optimizer golden record."""
    raw_question = sample[question_key]
    if isinstance(raw_question[0], list):
        question = "\n".join(q["content"] for q in raw_question[0])
    elif isinstance(raw_question[0], dict):
        question = "\n".join(q["content"] for q in raw_question)
    else:
        question = str(raw_question)

    # Merge all tool calls into a single flat dict
    # e.g. [{"weather": {...}}, {"hotel": {...}}] → {"weather": {...}, "hotel": {...}}
    merged_answer = {}
    for answer in sample["answer"]:
        for key, value in answer.items():
            key = reformat_tool_name(key)
            # Flatten nested single-element arrays (BFCL quirk)
            if isinstance(value, list) and len(value) == 1 and isinstance(value[0], list):
                value = value[0]
            # Float → string to match normalized tool schema
            if isinstance(value, float):
                value = str(value)
            elif isinstance(value, list):
                value = [str(v) if isinstance(v, float) else v for v in value]
            merged_answer[key] = value

    # answer must be a JSON object string, not a JSON array string
    return {
        "fields": {"question": question},
        "answer": json.dumps(merged_answer)   # "{...}" not "[{...}]"
    }

def load_bfcl_dataset(
    dataset_path: Path,
    n_train: int,
    n_test: int,
) -> Tuple[list, list, list]:
    """Load, sample, normalise, and split BFCL v3 data."""
    samples = read_bfcl_file(dataset_path)
    print(f"Total records in file: {len(samples)}")

    if samples:
        print(f"Sample keys: {list(samples[0].keys())}")

    n_total = min(n_train + n_test, len(samples))
    random.seed(42)
    sampled = random.sample(samples, n_total)

    tool_key     = detect_tool_key(sampled[0])
    question_key = detect_question_key(sampled[0])
    print(f"Tool key: '{tool_key}' | Question key: '{question_key}'")

    tools         = union_bfcl_tools(sampled, tool_key=tool_key)
    train_goldens = [build_golden(s, question_key=question_key) for s in sampled[:n_train]]
    test_goldens  = [build_golden(s, question_key=question_key) for s in sampled[n_train:]]
    return train_goldens, test_goldens, tools

# ── Load dataset ──────────────────────────────────────────────────────────────
dataset_path = Path(BFCL_DATASET)
train_goldens, test_goldens, tools = load_bfcl_dataset(
    dataset_path, N_TRAIN_SAMPLES, N_TEST_SAMPLES
)
print(f"Train goldens : {len(train_goldens)}")
print(f"Test goldens  : {len(test_goldens)}")
print(f"Unioned tools : {len(tools)}")
print(f"\nSample golden:\n{json.dumps(train_goldens[0], indent=2)}")

File size: 428,329 bytes
Loaded as concatenated JSON: 35 records
Total records in file: 35
Sample keys: ['id', 'question', 'function', 'answer']
Tool key: 'function' | Question key: 'question'
Train goldens : 25
Test goldens  : 10
Unioned tools : 10

Sample golden:
{
  "fields": {
    "question": "I'm planning a trip to Japan. I have 5000 US dollars and want to know how much that is in Japanese Yen. I'd also like to know the distance from Tokyo to Kyoto in kilometers. And while I'm researching Japanese companies, could you get me the latest stock price for Toyota?"
  },
  "answer": "{\"currency_conversion\": {\"amount\": [5000.0], \"from_currency\": [\"USD\", \"US Dollars\", \"US Dollar\"], \"to_currency\": [\"JPY\", \"Japanese Yen\"]}, \"calculate_distance\": {\"origin\": [\"Tokyo\"], \"destination\": [\"Kyoto\"], \"unit\": [\"km\", \"\"]}, \"get_stock_info\": {\"company\": [\"Toyota\", \"TM\"], \"metric\": [\"price\"]}}"
}


In [20]:
def build_tool_call_json_schema(tools: list) -> dict:
    """
    Build a JSON Schema for ResponseFormatJsonSchema from the unioned BFCL tool
    definitions produced in Step 3. Mirrors the golden-answer shape used
    throughout this notebook: a JSON object whose keys are tool names, and
    whose values are that tool's arguments object.
    """
    properties = {}
    for entry in tools:
        fn = entry["function"]
        properties[fn["name"]] = {
            "type": "object",
            "description": fn.get("description", ""),
            "properties": fn["parameters"].get("properties", {}),
        }

    return {
        "title": "ToolCallResponse",
        "type": "object",
        "description": "One or more tool calls, keyed by tool name.",
        "properties": properties,
        "minProperties": 1,
    }

tool_call_schema = build_tool_call_json_schema(tools)
print(f"Built JSON schema with {len(tool_call_schema['properties'])} tool properties.")

Built JSON schema with 10 tool properties.


---

## Step 4 — Upload Dataset Files to AI Core Storage

**What this step does:**
Serializes the four prepared objects to local JSON files, then uploads each one to a shared folder in AI Core's built-in dataset storage using the `/lm/dataset/files` endpoint.

**Files uploaded:**
| File | Contents |
|---|---|
| `bfcl_train.json` | 25 golden records used to train/refine the prompt |
| `bfcl_test.json` | 15 golden records used to evaluate candidate prompts |
| `bfcl_tools.json` | Union of all normalized tool definitions across samples |
| `bfcl_prompt_template.json` | The base prompt template spec |

All four files land in the same remote folder: `default/datasets/bfcl-optimizer/`

After uploading, the shared folder is registered as a single **dataset artifact** under the `genai-optimizations` scenario. The optimizer reads all files from this artifact folder.

> 💡 `get_or_create_artifact` checks for an existing artifact at the same URL before creating a new one — safe to re-run without creating duplicates.

In [7]:
# ── Serialize files for upload ────────────────────────────────────────────────
train_local  = "./bfcl_train.json"
test_local   = "./bfcl_test.json"
tools_local  = "./bfcl_tools.json"
prompt_local = "./bfcl_prompt_template.json"

with open(train_local,  "w") as f: json.dump(train_goldens,    f, indent=2)
with open(test_local,   "w") as f: json.dump(test_goldens,     f, indent=2)
with open(tools_local,  "w") as f: json.dump(tools,            f, indent=2)
with open(prompt_local, "w") as f: json.dump(prompt.model_dump(), f, indent=2)
print("Local files written.")



Local files written.


### 4.1 — The upload helper

`upload_file` does a raw HTTP `PUT` to AI Core's dataset-file endpoint. A few details worth noting:
- The remote path is URL-encoded (`quote(full_path, safe="")`) because it will contain slashes that need to survive being embedded in a URL path segment.
- `params={"overwrite": "true"}` means re-running this cell won't error out if the files already exist — it just replaces them, which is handy while you're iterating on the dataset.
- It returns the *folder* path (not the individual file path), since that's what gets registered as a single artifact in the next cell.

In [8]:
# ── Helpers: upload & artifact ────────────────────────────────────────────────
def upload_file(local_path: str, remote_subfolder: str, filename: str) -> str:
    """Upload file to AI Core dataset storage. Returns folder path 'default/<subfolder>'."""
    full_path    = f"default/{remote_subfolder}/{filename}"
    encoded_path = quote(full_path, safe="")
    url          = f"{client.ai_core_client.base_url}/lm/dataset/files/{encoded_path}"
    headers      = {**client.request_header, "Content-Type": "application/json"}
    with open(local_path, "rb") as f:
        res = requests.put(url, params={"overwrite": "true"}, headers=headers, data=f)
    print(f"  Upload [{filename}]: {res.status_code}")
    res.raise_for_status()
    return f"default/{remote_subfolder}"



### 4.2 — Uploading all four files and registering the artifact

This cell does two distinct things:

1. **Uploads** each of the four local files into the shared remote folder `default/datasets/bfcl-optimizer/`.
2. **Registers that folder as an artifact** via `get_or_create_artifact`. An artifact is AI Core's way of turning "a folder of files sitting in storage" into a first-class, referenceable object (with an `artifact_id`) that a configuration can bind as an input. The `ai://` prefix is AI Core's internal storage scheme.

`get_or_create_artifact` first **queries existing artifacts** in the scenario and reuses one if the same URL is already registered — that's why you may see `Reusing artifact [...]` printed instead of `Created artifact [...]` on repeated runs. Either way, hang onto the printed **Artifact ID** — it's used to build the optimization configuration in Step 6.

In [9]:
def get_or_create_artifact(name: str, folder_path: str, description: str) -> str:
    """Register folder as artifact. Returns artifact_id."""
    artifact_url = f"ai://{folder_path}"
    existing = client.ai_core_client.artifact.query(
        resource_group=resource_group, scenario_id=SCENARIO
    )
    for art in existing.resources:
        if art.url == artifact_url:
            print(f"  Reusing artifact [{name}]: {art.id}")
            return art.id
    resp = client.ai_core_client.artifact.create(
        name=name, kind=Artifact.Kind.DATASET,
        url=artifact_url, scenario_id=SCENARIO,
        resource_group=resource_group, description=description
    )
    print(f"  Created artifact [{name}]: {resp.id}")
    return resp.id

# ── Upload all files to shared folder ────────────────────────────────────────
REMOTE_SUBFOLDER = "datasets/bfcl-optimizer"
print("Uploading files...")
shared_folder = upload_file(train_local,  REMOTE_SUBFOLDER, "bfcl_train.json")
upload_file(test_local,   REMOTE_SUBFOLDER, "bfcl_test.json")
upload_file(tools_local,  REMOTE_SUBFOLDER, "bfcl_tools.json")
upload_file(prompt_local, REMOTE_SUBFOLDER, "bfcl_prompt_template.json")
print(f"Shared folder: {shared_folder}")

# ── Register dataset artifact ─────────────────────────────────────────────────
optimizer_artifact_id = get_or_create_artifact(
    name="bfcl-optimizer-data",
    folder_path=shared_folder,
    description="BFCL train/test goldens, tools, and prompt template"
)
print(f"Artifact ID: {optimizer_artifact_id}")


Uploading files...
  Upload [bfcl_train.json]: 201
  Upload [bfcl_test.json]: 201
  Upload [bfcl_tools.json]: 201
  Upload [bfcl_prompt_template.json]: 201
Shared folder: default/datasets/bfcl-optimizer
  Reusing artifact [bfcl-optimizer-data]: e4a97767-8130-4e8c-9ded-b0313eb7d4ad
Artifact ID: e4a97767-8130-4e8c-9ded-b0313eb7d4ad


---

## Step 5 — Create and Register the Base Prompt Template

**What this step does:**
Pushes the base prompt template to the Prompt Registry under the `genai-optimizations` scenario.

The base prompt is intentionally minimal:
- **System:** `"You are a helpful assistant."`
- **User:** `{{?question}}`

**Why start so minimal?** The whole point of prompt optimization is to let the optimizer discover a *better* prompt than you'd write by hand — starting from a deliberately weak, generic prompt gives the optimizer maximum room to add structure (JSON-only output instructions, tool schemas, reasoning steps, etc.) and lets you clearly see, in Step 9, how much value the optimization actually added.

The optimizer takes this as its starting point and iteratively rewrites it during execution. When done, the final refined prompt is saved back to the registry under the name specified in `targetPromptMapping` (e.g., `bfcl-tool-optimized-gemini25:0.0.1`).

**What does `{{?question}}` mean?** It's a template placeholder — at inference time, the actual user question gets substituted in for `{{?question}}`. You'll see this same substitution pattern used later in `run_inference` (`user_content = user_template.replace("{{?question}}", question)`).

> 💡 A `409` response means the prompt already exists — this is safe and the existing version is reused automatically.

In [10]:
# ── Push prompt to registry ───────────────────────────────────────────────────
def push_prompt(spec: PromptTemplateSpec, name: str, version: str, scenario: str):
    url  = f"{client.ai_core_client.base_url}/lm/promptTemplates"
    body = {"name": name, "version": version, "scenario": scenario, "spec": spec.model_dump()}
    res  = requests.post(
        url,
        headers={**client.request_header, "Content-Type": "application/json"},
        json=body
    )
    print(f"Prompt registry: {res.status_code} — {res.json().get('message', '')}")
    if res.status_code == 409:
        print("Prompt already exists — reusing.")
        return {"name": name, "version": version}
    res.raise_for_status()
    return res.json()

push_prompt(prompt, PROMPT_NAME, PROMPT_VERSION, SCENARIO)

Prompt registry: 200 — Prompt updated successfully.


{'message': 'Prompt updated successfully.',
 'id': '235d49b4-0572-4926-9b77-4d0977559803',
 'scenario': 'genai-optimizations',
 'name': 'bfcl-tool-base',
 'version': '0.0.1'}

**Reading the output:** A `200`/`201` with `"Prompt updated successfully."` (or a `409` handled by the `if` branch) confirms the base prompt `bfcl-tool-base:0.0.1` now exists in the registry. The returned `id` is the prompt template's registry ID — you'll see it again later when listing all templates in Step 8.

---

## Step 6 — Register an Optimization Configuration

**What this step does:**
Creates the optimization configuration that links all inputs together — the artifact, base prompt, reference model, target model, and metric — into one executable setup. Note that **creating a configuration does not run anything yet** — think of it as saving a recipe; Step 7 is when you actually "cook" it by triggering an execution.

**15 parameter bindings explained:**

| Parameter | Value | Purpose |
|---|---|---|
| `optimizationMetric` | `JSON_Match` | Evaluates tool call JSON accuracy against the golden answer |
| `basePrompt` | `genai-optimizations/bfcl-tool-base:0.0.1` | Starting prompt in the registry |
| `baseModel` | `gpt-4o:2024-08-06` | Reference/teacher model |
| `targetModels` | `gemini-2.5-pro:001` | Model to optimize for |
| `targetPromptMapping` | `gemini-2.5-pro:001=bfcl-tool-optimized-gemini25:0.0.1` | Which registry name+version the final optimized prompt for this target model will be saved as |
| `trainDataset` | `bfcl_train.json` | Training file in the artifact folder |
| `testDataset` | `bfcl_test.json` | Evaluation file in the artifact folder |
| `maximize` | `true` | Higher metric score = better |
| `correctnessCutoff` | `none` | No hard pass/fail threshold applied — every candidate is scored on a continuous scale |
| `includeFewShotExamples` | `false` | No few-shot injection — the optimizer only rewrites instructions, not example Q&A pairs, into the prompt |
| `promptTemplateScope` | `tenant` | The resulting prompt template is visible tenant-wide |
| `prototypeMode` | `false` | Runs the full, standard optimization process |
| `fieldEvaluationMetrics` | `none` | No per-field (sub-answer) scoring breakdown requested |
| `modelParams` | `none` | No extra model parameters (temperature, max tokens, etc.) overridden |
| `customMetricId` | `none` | Not used here — this notebook relies on the built-in `JSON_Match` metric rather than a custom LLM-as-a-judge one |

**Why an `InputArtifactBinding`?** Parameters (`ParameterBinding`) are simple key-value strings, but the *dataset itself* is a folder of files, not a string — so it's passed separately as an `InputArtifactBinding`, referencing the `artifact_id` you got back in Step 4. The configuration only knows *filenames* (`trainDataset`/`testDataset`); it resolves those filenames against the bound artifact folder at execution time.

> 💡 `create_config` checks for an existing configuration with identical parameters before creating a new one — safe to re-run.

In [11]:
# ── Create configuration ──────────────────────────────────────────────────────
def create_config(
    metric: str,
    reference_model: str,
    targets: dict,
    train_filename: str,
    test_filename: str,
    prompt_artifact_id: str,
    prompt_name: str,
    prompt_version: str,
    scenario: str,
) -> str:
    base_prompt = f"{scenario}/{prompt_name}:{prompt_version}"

    input_parameters = [
        ParameterBinding(key="optimizationMetric",     value=metric),
        ParameterBinding(key="basePrompt",             value=base_prompt),
        ParameterBinding(key="baseModel",              value=reference_model),
        ParameterBinding(key="targetModels",           value=",".join(targets.keys())),
        ParameterBinding(
            key="targetPromptMapping",
            value=",".join(f"{k}={v}" for k, v in targets.items())
        ),
        ParameterBinding(key="trainDataset",           value=train_filename),
        ParameterBinding(key="testDataset",            value=test_filename),
        ParameterBinding(key="maximize",               value="true"),
        ParameterBinding(key="correctnessCutoff",      value="none"),
        ParameterBinding(key="includeFewShotExamples", value="false"),
        ParameterBinding(key="promptTemplateScope",    value="tenant"),
        ParameterBinding(key="prototypeMode",          value="false"),
        ParameterBinding(key="fieldEvaluationMetrics", value="none"),
        ParameterBinding(key="modelParams",            value="none"),
        ParameterBinding(key="customMetricId",         value="none"),
    ]

    input_artifacts = [
        InputArtifactBinding(key="prompt-data", artifact_id=prompt_artifact_id)
    ]

    params_dict = {p.key: p.value for p in input_parameters}

    try:
        existing = client.ai_core_client.configuration.query(
            scenario_id=SCENARIO, resource_group=resource_group
        )
        for conf in existing.resources:
            if {p.key: p.value for p in conf.parameter_bindings} == params_dict:
                print(f"Reusing configuration: {conf.id}")
                return conf.id
    except Exception as e:
        print(f"Could not query configs: {e}")

    resp = client.ai_core_client.configuration.create(
        name="bfcl-tool-config",
        scenario_id=SCENARIO,
        executable_id=SCENARIO,
        resource_group=resource_group,
        parameter_bindings=input_parameters,
        input_artifact_bindings=input_artifacts,
    )
    print(f"Created configuration: {resp.id}")
    return resp.id

configuration_id = create_config(
    metric=METRIC,
    reference_model=REFERENCE_MODEL,
    targets=TARGET_MODELS,
    train_filename="bfcl_train.json",
    test_filename="bfcl_test.json",
    prompt_artifact_id=optimizer_artifact_id,
    prompt_name=PROMPT_NAME,
    prompt_version=PROMPT_VERSION,
    scenario=SCENARIO,
)
print(f"Configuration ID: {configuration_id}")

Reusing configuration: 187c93a0-636b-4fe7-b77b-c89f8c2703d7
Configuration ID: 187c93a0-636b-4fe7-b77b-c89f8c2703d7


**Reading the output:** you'll get a `Configuration ID` (either freshly created, or reused if an identical configuration already existed — as shown here, `Reusing configuration: ...`). Keep this ID handy — it's the single thing Step 7 needs to actually launch the optimization run.

---

## Step 7 — Run the Prompt Optimization Execution & Monitor Progress

**What this step does:**
Triggers the optimization job and polls its status every 30 seconds until completion.

**Configuration vs. Execution, revisited:** the configuration you created in Step 6 is a reusable *template* for a run — you could trigger multiple executions from the same configuration (e.g. to get a second opinion, or after fixing a data issue). Each execution gets its own ID and runs independently.

**Execution status transitions:**
```
UNKNOWN → RUNNING (progress=1/100) → RUNNING (progress=32/100) → RUNNING (progress=70/100) → COMPLETED (progress=100/100)
```

**Expected duration:** ~20–30 minutes for 25 train + 15 test samples. Behind the scenes, the optimizer is generating candidate prompt rewrites, running each candidate against your test set through the reference/target models, scoring the results with `JSON_Match`, and iterating — so runtime scales with dataset size and number of candidate iterations.

**If execution fails (`FAILED` / `DEAD`):**
The cell automatically fetches and prints the execution logs to help diagnose the issue — this saves you a trip to the AI Core console/UI to dig up error details.

> ⚠️ The cell will keep polling until a terminal status is reached. You can interrupt it with `Kernel → Interrupt` if needed — the execution continues running on AI Core even after interruption, since it's a server-side job, not a local process. You can always come back and re-poll `execution_id` later instead of re-running the whole cell from scratch.

In [12]:
# ── Execute ───────────────────────────────────────────────────────────────────
execution = client.ai_core_client.execution.create(
    configuration_id=configuration_id,
    resource_group=resource_group,
)
execution_id = execution.id
print(f"Execution ID: {execution_id}")



Execution ID: e03a17fd2fd39cc7


The execution has been triggered and is now running server-side on AI Core. The next cell polls its status in a loop — this is the part of the notebook that takes the longest, so feel free to step away and come back.

In [13]:
TERMINAL_STATES = {"COMPLETED", "FAILED", "DEAD", "STOPPED"}

while True:
    status = client.ai_core_client.execution.get(
        execution_id=execution_id, resource_group=resource_group
    )
    # Safely convert enum to string
    status_str = status.status.value if hasattr(status.status, "value") else str(status.status)
    print(f"[{time.strftime('%H:%M:%S')}] {status_str}", end="")

    if hasattr(status, "status_details") and status.status_details:
        progress = status.status_details.get("progress", "")
        print(f"  progress={progress}", end="")
    print()

    if status_str in TERMINAL_STATES:
        if status_str != "COMPLETED":
            try:
                logs = client.ai_core_client.execution.get_logs(
                    execution_id=execution_id, resource_group=resource_group
                )
                print("── Execution logs ──")
                for log in logs.data:
                    print(log.msg)
            except Exception as e:
                print(f"Could not fetch logs: {e}")
        break

    time.sleep(30)

print(f"\nFinal status: {status_str}")

[23:55:47] UNKNOWN
[23:56:18] UNKNOWN
[23:56:48] RUNNING  progress=0/100
[23:57:19] RUNNING  progress=0/100
[23:57:50] RUNNING  progress=0/100
[23:58:21] RUNNING  progress=0/100
[23:58:52] RUNNING  progress=0/100
[23:59:23] RUNNING  progress=0/100
[23:59:55] RUNNING  progress=1/100
[00:00:26] RUNNING  progress=1/100
[00:00:56] RUNNING  progress=1/100
[00:01:28] RUNNING  progress=1/100
[00:01:59] RUNNING  progress=1/100
[00:02:29] RUNNING  progress=1/100
[00:03:01] RUNNING  progress=1/100
[00:03:32] RUNNING  progress=1/100
[00:04:03] RUNNING  progress=1/100
[00:04:35] RUNNING  progress=1/100
[00:05:06] RUNNING  progress=1/100
[00:05:37] RUNNING  progress=1/100
[00:06:09] RUNNING  progress=32/100
[00:06:40] RUNNING  progress=32/100
[00:07:11] RUNNING  progress=32/100
[00:07:42] RUNNING  progress=32/100
[00:08:13] RUNNING  progress=32/100
[00:08:44] RUNNING  progress=32/100
[00:09:16] RUNNING  progress=32/100
[00:09:46] RUNNING  progress=32/100
[00:10:17] RUNNING  progress=32/100
[00:10:4

**Reading the output:** each line is one poll (roughly every 30 seconds), showing the timestamp, status, and progress percentage. `Final status: COMPLETED` means the optimizer finished successfully and a new, refined prompt has been written back to the Prompt Registry — that's what Step 8 retrieves next. If you instead see `FAILED`/`DEAD`/`STOPPED`, read the printed execution logs carefully; common culprits include a model not being available in your region, a malformed dataset file, or a metric misconfiguration.

---

## Step 8 — Fetch the Full Optimized Prompt by ID

**What this step does:**
Retrieves the complete optimized prompt template from the Prompt Registry by its ID.

**Two-step lookup:** the Prompt Registry doesn't offer a "get by name" shortcut in this flow, so we first **list every prompt template** in the tenant (next cell) to find the `id` that corresponds to the name you're looking for (e.g. `bfcl-tool-optimized-gemini25` or your own `targetPromptMapping` value from Step 6), and then **fetch that specific ID** (the cell after) to see its full content.

**Replace `optimized_id`** with the actual ID of your optimized prompt found in the listing below — the ID shown in this notebook is from a specific past run and won't match your own tenant.

The optimized prompt will be significantly more detailed than the base `"You are a helpful assistant."` — it will contain:
- A structured-output parser role definition
- Last-of-type selection rules for duplicate function calls
- Strict JSON output constraints (no markdown, no backticks)
- Full tool schemas with normalization rules for all 10 functions
- Reasoning steps and concrete exemplars

**Connecting back to the Response Formatting primer:** notice that the optimizer achieves structured output purely through *prompt wording* — instructions like "no markdown, no backticks" are model-facing suggestions, not API-enforced constraints. If you wanted an API-level guarantee on top of this, you'd pair this exact prompt with a `response_format=json_schema` setting (see the primer after Step 1) built from the tool schemas in `bfcl_tools.json`.

In [15]:
url = f"{client.ai_core_client.base_url}/lm/promptTemplates"
res = requests.get(url, headers=client.request_header)
templates = res.json()
for t in templates.get("resources", []):
    print(f"  name={t['name']}  version={t['version']}  id={t['id']}")

  name=multi_task_withRG  version=1.1.3  id=35f5c235-b949-49bb-854e-cca0913086ab
  name=expand_text  version=1.1.2  id=c050ab3a-1653-41d5-8d76-1c2f0a4457fd
  name=multi_task_withRG  version=1.1.2  id=9b502632-8ddc-4d44-bc0c-78044f8ca63b
  name=multi_task  version=1.1.1  id=fca6185e-e340-4781-9d13-00e32f510674
  name=facility-json-template  version=1.0.0  id=27ac3122-9b6a-4baa-a7a2-c3670cea83b2
  name=prompt-registry-eval-demo  version=1.0.0  id=9f47e745-5c87-46df-b917-8e7dc829f47c
  name=evalPromptTemplateConfig-227e9e3  version=1.0.0  id=d4416a6f-8f45-458e-81a2-f3d2f346a26e
  name=evalPromptTemplateConfig-8eb5f38  version=1.0.0  id=87faced1-57b6-4445-aa3f-6294f1a5a8b0
  name=evalPromptTemplateConfig-25152b4  version=1.0.0  id=24dfbaad-9609-4a69-abf9-e3053aa90bf2
  name=evalPromptTemplateConfig-de19a80  version=1.0.0  id=5ac8367e-fc1d-4aed-87ae-cb5166112b1c
  name=evalPromptTemplateConfig-fa00f03  version=1.0.0  id=ef294c8f-ded7-4ddf-a3ec-aa415efc7300
  name=evalPromptTemplateConfig-f5

**Reading the output:** this is simply a directory listing of every prompt template in your tenant — scan it for the name you registered as the optimizer's output target (set in `targetPromptMapping` back in Step 6), and copy its `id` value into the next cell.

> 💡 Note: the specific `id` used in the next cell below (`d4416a6f-...`, name `evalPromptTemplateConfig-227e9e3`) is an example from a different prompt in this listing, not the BFCL optimization output — when you run this yourself, make sure to copy the ID that actually corresponds to *your* target prompt name (e.g. `bfcl-tool-optimized-gemini25` or whatever you set in Step 6), not just copy the ID shown here verbatim.

In [16]:
# ── Fetch optimized prompt template ──────────────────────────────────────────
# Replace with the actual ID of bfcl-tool-optimized-gemini25 from the listing above
optimized_id = "a5c8f379-b90f-42ec-a176-6f7f9ae44f6c"

url = f"{client.ai_core_client.base_url}/lm/promptTemplates/{optimized_id}"
res = requests.get(url, headers=client.request_header)
print(f"Status: {res.status_code}")
optimized = res.json()
print(json.dumps(optimized, indent=2))

Status: 200
{
  "id": "a5c8f379-b90f-42ec-a176-6f7f9ae44f6c",
  "name": "bfcl-tool-optimized-gemini25",
  "version": "0.0.1",
  "scenario": "genai-optimizations",
  "creationTimestamp": "2026-07-01T18:50:10.130000",
  "managedBy": "imperative",
  "isVersionHead": true,
  "spec": {
    "template": [
      {
        "role": "system",
        "content": "You are a structured-output extraction and tool-call planning agent. Your role is to convert any natural-language user question into a single strict JSON object that specifies which available tools to call and with what arguments. Do not answer questions with real-world results. Do not include explanations, disclaimers, or markdown in your final output. Output only the JSON object.\n\nObjectives and output contract:\n- Identify all supported intents present in the question and map them to tool names from the catalog below.\n- For each used tool, produce an arguments object with normalized, deduplicated values.\n- Emit exactly one strict J

### 9.1 — Finding a live orchestration deployment

Live inference in AI Core runs through a **deployed** orchestration scenario (a running service with its own URL), not just an API key + model name. This cell lists every deployment in your tenant so you can find one with `scenario=orchestration` and `status=RUNNING`, then copies its URL into `ORCHESTRATION_DEPLOYMENT_URL` in the next cell. If you don't have a running orchestration deployment yet, you'd need to create one first via the AI Core console/API before this step will work.

In [17]:
# Find the running orchestration deployment URL
url = f"{client.ai_core_client.base_url}/lm/deployments"
res = requests.get(url, headers=client.request_header)
for d in res.json().get("resources", []):
    print(f"id={d.get('id')}  scenario={d.get('scenarioId'):30s}  status={d.get('status'):10s}  url={d.get('deploymentUrl')}")

id=ddee9146ce463a8b  scenario=orchestration                   status=RUNNING     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/ddee9146ce463a8b
id=d46fa82110bbc6c8  scenario=foundation-models               status=RUNNING     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d46fa82110bbc6c8
id=d6fa93ca356f105a  scenario=foundation-models               status=STOPPED     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d6fa93ca356f105a
id=d0d13204b3862077  scenario=foundation-models               status=RUNNING     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d0d13204b3862077
id=d0048ce66805cd7a  scenario=foundation-models               status=RUNNING     url=wss://realtime.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d0048ce66805cd7a
id=dc0c40f766de1b52  scenario=or

In [21]:
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.service import OrchestrationService

# ── Step 1: Fetch both prompt templates from the registry ─────────────────────
def get_prompt_template(template_id: str) -> dict:
    url = f"{client.ai_core_client.base_url}/lm/promptTemplates/{template_id}"
    res = requests.get(url, headers=client.request_header)
    res.raise_for_status()
    return res.json()

def extract_messages(template: dict) -> dict:
    messages = {}
    for msg in template.get("spec", {}).get("template", []):
        messages[msg["role"]] = msg["content"]
    return messages

# Look up IDs by name from the registry
url = f"{client.ai_core_client.base_url}/lm/promptTemplates"
res = requests.get(url, headers=client.request_header)
all_templates = {
    f"{t['name']}:{t['version']}": t["id"]
    for t in res.json().get("resources", [])
}

base_id      = all_templates.get("bfcl-tool-base:0.0.1")
optimized_id = all_templates.get("bfcl-tool-optimized-gemini25:0.0.1")

base_template      = get_prompt_template(base_id)
optimized_template = get_prompt_template(optimized_id)

base_messages      = extract_messages(base_template)
optimized_messages = extract_messages(optimized_template)

print("✅ Loaded base and optimized prompt templates.")
print(f"Base system prompt      : {base_messages['system'][:80]}...")
print(f"Optimized system prompt : {optimized_messages['system'][:80]}...")


# ── Step 2: run_inference using OrchestrationService ────────────────────────
# Paste your orchestration deployment URL here
ORCHESTRATION_DEPLOYMENT_URL = "https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/ddee9146ce463a8b"

from gen_ai_hub.orchestration.models.response_format import ResponseFormatJsonSchema

def run_inference(system_prompt: str, user_template: str, question: str, model_name: str,
                   use_response_format: bool = False) -> str:
    user_content = user_template.replace("{{?question}}", question)

    template_kwargs = {
        "messages": [
            SystemMessage(system_prompt),
            UserMessage(user_content),
        ]
    }

    if use_response_format:
        template_kwargs["response_format"] = ResponseFormatJsonSchema(
            name="tool_call_response",
            description="Structured tool call output, keyed by tool name.",
            schema=tool_call_schema,
        )

    config = OrchestrationConfig(
        llm=LLM(name=model_name),
        template=Template(**template_kwargs),
    )

    service  = OrchestrationService(api_url=ORCHESTRATION_DEPLOYMENT_URL, config=config)
    response = service.run()
    return response.module_results.llm.choices[0].message.content
# ── Step 3: Compare prompts ───────────────────────────────────────────────────
def clean_json_output(text: str) -> str:
    """Strip markdown code fences that models sometimes wrap around JSON."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()
    return text


def compare_prompts(question: str, model_name: str = "gemini-2.5-pro"):
    print("\n" + "=" * 70)
    print(f"QUESTION:\n{question}")
    print("=" * 70)

    print("\n📌 BASE PROMPT OUTPUT:")
    print("-" * 40)
    try:
        base_output = run_inference(
            system_prompt=base_messages["system"],
            user_template=base_messages["user"],
            question=question,
            model_name=model_name,
        )
        print(base_output)
    except Exception as e:
        base_output = f"ERROR: {e}"
        print(base_output)

    print("\n✅ OPTIMIZED PROMPT OUTPUT:")
    print("-" * 40)
    try:
        optimized_output = run_inference(
            system_prompt=optimized_messages["system"],
            user_template=optimized_messages["user"],
            question=question,
            model_name=model_name,
        )
        print(optimized_output)
    except Exception as e:
        optimized_output = f"ERROR: {e}"
        print(optimized_output)

    print("\n📊 COMPARISON:")
    print("-" * 40)
    for label, output in [("BASE", base_output), ("OPTIMIZED", optimized_output)]:
        cleaned = clean_json_output(output)
        try:
            parsed = json.loads(cleaned)
            tools  = list(parsed.keys())
            print(f"{label:12s} → valid JSON ✅ | tools called: {tools}")
        except json.JSONDecodeError:
            print(f"{label:12s} → invalid JSON ❌ | raw: {output[:120]}")

    print("\n📈 VERDICT:")
    print("-" * 40)
    base_valid      = True
    optimized_valid = True
    try:
        json.loads(clean_json_output(base_output))
    except Exception:
        base_valid = False
    try:
        json.loads(clean_json_output(optimized_output))
    except Exception:
        optimized_valid = False

    if not base_valid and optimized_valid:
        print("🏆 Optimization WIN — base gave prose, optimized gave structured JSON")
    elif base_valid and optimized_valid:
        print("✅ Both valid JSON — compare tool accuracy above")
    elif base_valid and not optimized_valid:
        print("⚠️  Base was valid but optimized was not — check prompt")
    else:
        print("❌ Both invalid — check model or deployment")

    return base_output, optimized_output


# ── Step 4: Run all comparisons ───────────────────────────────────────────────
question = "What is the weather in Tokyo for the next 3 days in celsius?"

without_rf = run_inference(
    system_prompt=optimized_messages["system"],
    user_template=optimized_messages["user"],
    question=question,
    model_name="gemini-2.5-pro",
    use_response_format=False,
)

with_rf = run_inference(
    system_prompt=optimized_messages["system"],
    user_template=optimized_messages["user"],
    question=question,
    model_name="gemini-2.5-pro",
    use_response_format=True,
)

print("WITHOUT response_format:\n", without_rf)
print("\nWITH response_format:\n", with_rf)

✅ Loaded base and optimized prompt templates.
Base system prompt      : You are a helpful assistant....
Optimized system prompt : You are a structured-output extraction and tool-call planning agent. Your role i...
WITHOUT response_format:
 ```json
{
  "weather_forecast": {
    "days": [
      3
    ],
    "location": [
      "Tokyo",
      "Tokyo, Japan"
    ],
    "units": [
      "celsius"
    ]
  }
}
```

WITH response_format:
 {
  "weather_forecast": {
    "days": 3,
    "location": "Tokyo",
    "units": "celsius"
  }
}
